# KNN phân loại sản phẩm: có purchase trong 7 ngày tới hay không

**Câu hỏi.** Từ hành vi của một sản phẩm trong 14 ngày trước mốc `t0`, có dự đoán được sản phẩm đó có ít nhất một purchase trong 7 ngày sau `t0` hay không?

**Giả thuyết kiểm chứng được.** Trên tập test (ảnh chụp thời gian sau), KNN có F1 và PR-AUC của lớp dương cao hơn baseline lớp đa số và baseline luật "đã từng có purchase trong lịch sử". Có so sánh thêm với Logistic Regression (Spark MLlib) làm mốc tham chiếu. KNN thua thì báo cáo đúng như vậy.

**Đơn vị quan sát và nhãn.** 1 dòng = 1 `product_id` tại một ảnh chụp, do job Java `ProductLabelJob` (A8) tạo trên HDFS từ curated Parquet:
- đặc trưng chỉ dùng sự kiện trong `[t0 − 14 ngày, t0)` (A7 + `recent_view_share`), lọc `views ≥ 20`;
- nhãn = 1 nếu có purchase trong `[t0, t0 + 7 ngày)`.

**Chống rò rỉ dữ liệu.** Train là ảnh chụp tại `t0 = 2019-10-15`, test là ảnh chụp tại `t0 = 2019-10-25`; cửa sổ nhãn của train kết thúc trước `t0` của test. Job A8 kiểm tra `max(event_time lịch sử) < t0` và độ phủ dữ liệu của cửa sổ nhãn. Scaler chỉ fit trên phần *fit* của train. K, ngưỡng và Logistic Regression chỉ chọn trên phần *validation* (20% train, chia theo hash `product_id`). Test chỉ dùng **một lần** ở cuối.

**Về công cụ.** Spark MLlib **không có** bộ phân loại KNN. Spark tạo đặc trưng/nhãn (Java) và chuẩn hóa (`StandardScaler`); KNN ở đây là tìm láng giềng chính xác bằng numpy (`ml_common.knn_vote_share`). KNN là *lazy learner*: "mô hình" = tập huấn luyện đã chuẩn hóa + K + ngưỡng. `vote_share` là tỷ lệ phiếu, không phải xác suất đã hiệu chỉnh.

**Chỉ số.** Lớp dương có thể chiếm thiểu số nên accuracy dễ gây hiểu sai; chỉ số chính là precision/recall/F1 lớp dương, PR-AUC (average precision) và balanced accuracy; vẫn báo accuracy và confusion matrix.

**Quy tắc chọn (cố định trước khi chạy).** Với mỗi K: ngưỡng = giá trị `vote_share` cho F1 validation cao nhất (hòa → ngưỡng lớn hơn). Chọn K có F1 validation cao nhất (hòa → K nhỏ hơn).

## 1. Tham số (biến môi trường, mặc định trong ngoặc)

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import ml_common as mc

TAG = mc.param("ML_TAG", "d3")                                   # d3 = cả tháng 10/2019
LABELS_RUN_ID = mc.param("ML_LABELS_RUN_ID") or mc.pipeline_run_id(TAG, "labels")
K_VALUES = [int(k) for k in mc.param("ML_KNN_K_VALUES", "3,5,7,9,15").split(",")]
SEED = int(mc.param("ML_SEED", "21"))
MAX_TRAIN = int(mc.param("ML_KNN_MAX_TRAIN", "50000"))           # trần số dòng train giữ trong mô hình (RAM, thời gian dự đoán)
VALIDATION_PERCENT = 20
BOOTSTRAP = 1000
RUN_ID = mc.new_run_id(TAG)
OUT = f"/data/ecommerce/ml/knn/run_id={RUN_ID}"
LOCAL = mc.RESULTS_ML / RUN_ID
print("labels run:", LABELS_RUN_ID, "| knn run:", RUN_ID, "| K:", K_VALUES)

labels run: 20261008-195059-97bb8c2-dk3 | knn run: 20261008-201830-97bb8c2-dk3 | K: [3, 5, 7, 9, 15]


## 2. SparkSession (local) đọc HDFS

In [2]:
spark = mc.session(f"knn-classifier-{RUN_ID}")
ENV = mc.environment(spark)
trace = mc.Trace("knn-classifier", RUN_ID, {"labelsRunId": LABELS_RUN_ID, "kValues": K_VALUES, "seed": SEED,
                                             "maxTrain": MAX_TRAIN, "validationPercent": VALIDATION_PERCENT})
ENV

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/08 20:18:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'sparkVersion': '4.0.4',
 'javaVersion': '21.0.9',
 'hadoopClientVersion': '3.4.1',
 'pythonVersion': '3.12.3',
 'master': 'local[2]',
 'driverMemory': '1g',
 'gitSha': '97bb8c2'}

## 3. Đọc A8, kiểm tra rò rỉ và tỷ lệ lớp

In [3]:
from pyspark.sql import functions as F
LABELS_PATH = mc.hdfs(f"{mc.LABELS_ROOT}/run_id={LABELS_RUN_ID}")
data = spark.read.parquet(LABELS_PATH)
missing = [c for c in mc.LABEL_FEATURES + ["label", "split", "t0"] if c not in data.columns]
assert not missing, f"Thiếu cột: {missing}"
labels_run = json.loads(mc.read_hdfs_text(spark, f"{mc.LABELS_ROOT}/run_id={LABELS_RUN_ID}/_run.json"))
leakage = labels_run["metrics"]["leakageChecks"]
assert all(v["historyBeforeT0"] for v in leakage.values())
print(json.dumps({"params": labels_run["params"], "leakageChecks": leakage}, indent=2))
class_table = (data.groupBy("split", "t0").agg(F.count(F.lit(1)).alias("rows"), F.sum("label").alias("positives"))
               .withColumn("positive_rate", F.round(F.col("positives") / F.col("rows"), 4)).orderBy("t0").toPandas())
class_table

{
  "params": {
    "curatedRunId": "20261008-185644-97bb8c2-dk3",
    "trainT0": "2019-10-15",
    "testT0": "2019-10-25",
    "featureDays": 14,
    "labelDays": 7,
    "minViews": 20,
    "output": "/data/ecommerce/features/product_label/run_id=20261008-195059-97bb8c2-dk3"
  },
  "leakageChecks": {
    "train": {
      "t0": "2019-10-15",
      "historyMinEventTime": "2019-10-01T00:00:00Z",
      "historyMaxEventTime": "2019-10-14T23:59:59Z",
      "labelMinEventTime": "2019-10-15T00:00:00Z",
      "labelMaxEventTime": "2019-10-21T23:59:59Z",
      "historyBeforeT0": true
    },
    "test": {
      "t0": "2019-10-25",
      "historyMinEventTime": "2019-10-11T00:00:00Z",
      "historyMaxEventTime": "2019-10-24T23:59:59Z",
      "labelMinEventTime": "2019-10-25T00:00:00Z",
      "labelMaxEventTime": "2019-10-31T23:59:59Z",
      "historyBeforeT0": true
    }
  }
}


,split,t0,rows,positives,positive_rate
0,train,2019-10-15,60876,17057,0.2802
1,test,2019-10-25,64254,16278,0.2533


## 4. Chia fit/validation (hash `product_id`, chỉ trong ảnh chụp train) và chuẩn hóa
`StandardScaler` (Spark MLlib) chỉ fit trên phần *fit*, sau đó áp cho validation và test. Không dùng category làm đặc trưng.

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array
bucket = F.pmod(F.xxhash64("product_id", F.lit(SEED)), F.lit(100))
train = data.where("split = 'train'").withColumn("part", F.when(bucket < VALIDATION_PERCENT, "validation").otherwise("fit"))
test = data.where("split = 'test'").withColumn("part", F.lit("test"))
with trace.stage("fit_scaler"):
    prep = Pipeline(stages=[VectorAssembler(inputCols=mc.LABEL_FEATURES, outputCol="raw_features"),
                            StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)]
                   ).fit(train.where("part = 'fit'"))
scaler = prep.stages[-1]

def to_numpy(df):
    pdf = (prep.transform(df).select("product_id", "label", "purchases", vector_to_array("features").alias("v"))
           .orderBy("product_id").toPandas())
    return pdf["product_id"].to_numpy(), np.array(pdf["v"].tolist()), pdf["label"].to_numpy().astype(int), pdf["purchases"].to_numpy()

with trace.stage("collect"):
    fit_ids, X_fit, y_fit, past_fit = to_numpy(train.where("part = 'fit'"))
    val_ids, X_val, y_val, past_val = to_numpy(train.where("part = 'validation'"))
    test_ids, X_test, y_test, past_test = to_numpy(test)
fit_rows_before_cap = len(y_fit)
if len(y_fit) > MAX_TRAIN:   # lấy mẫu ngẫu nhiên có seed, giữ nguyên thứ tự product_id
    keep = np.sort(np.random.default_rng(SEED).choice(len(y_fit), MAX_TRAIN, replace=False))
    fit_ids, X_fit, y_fit, past_fit = fit_ids[keep], X_fit[keep], y_fit[keep], past_fit[keep]
pd.DataFrame({"rows": [fit_rows_before_cap, len(y_fit), len(y_val), len(y_test)],
              "positive_rate": [np.nan, y_fit.mean(), y_val.mean(), y_test.mean()]},
             index=["fit (trước trần)", "fit (dùng)", "validation", "test"])

[20:18:58] bắt đầu: fit_scaler


[20:19:00] xong: fit_scaler (1721 ms)
[20:19:00] bắt đầu: collect


[20:19:08] xong: collect (7828 ms)


,rows,positive_rate
fit (trước trần),48721,NaN
fit (dùng),48721,0.280782
validation,12155,0.277828
test,64254,0.253338


## 5. Quét K trên validation

In [5]:
rows, val_scores = [], {}
with trace.stage("k_sweep"):
    for k in K_VALUES:
        t0 = time.perf_counter()
        scores = mc.knn_vote_share(X_fit, y_fit, X_val, k)
        threshold = mc.best_threshold(y_val, scores, [i / k for i in range(1, k + 1)])
        m = mc.binary_metrics(y_val, scores >= threshold)
        rows.append({"k": k, "threshold": threshold, "pr_auc": mc.average_precision(y_val, scores),
                     "f1_at_0.5": mc.binary_metrics(y_val, scores >= 0.5)["f1"], **m,
                     "predict_ms": round((time.perf_counter() - t0) * 1000)})
        val_scores[k] = scores
        print(f"k={k:2d} threshold={threshold:.3f} F1={m['f1']:.4f} PR-AUC={rows[-1]['pr_auc']:.4f} {rows[-1]['predict_ms']} ms")
sweep = pd.DataFrame(rows).set_index("k")
best_k = int(sweep.assign(r=sweep.f1.round(6)).query("r == r.max()").index.min())
best_threshold = float(sweep.loc[best_k, "threshold"])
print(f"K chọn = {best_k}, ngưỡng vote_share = {best_threshold:.3f}")
sweep[["threshold", "precision", "recall", "f1", "f1_at_0.5", "pr_auc", "balanced_accuracy", "accuracy", "predict_ms"]].round(4)

[20:19:08] bắt đầu: k_sweep


k= 3 threshold=0.667 F1=0.5739 PR-AUC=0.5578 56484 ms


k= 5 threshold=0.400 F1=0.5989 PR-AUC=0.6159 45297 ms


k= 7 threshold=0.429 F1=0.6122 PR-AUC=0.6437 54598 ms


k= 9 threshold=0.444 F1=0.6164 PR-AUC=0.6616 41870 ms


k=15 threshold=0.400 F1=0.6268 PR-AUC=0.6860 41630 ms
[20:23:08] xong: k_sweep (239918 ms)


K chọn = 15, ngưỡng vote_share = 0.400


,threshold,precision,recall,f1,f1_at_0.5,pr_auc,balanced_accuracy,accuracy,predict_ms
k,,,,,,,,,
3,0.6667,0.6299,0.5271,0.5739,0.5739,0.5578,0.7040,0.7826,56484
5,0.4000,0.5341,0.6817,0.5989,0.5856,0.6159,0.7265,0.7464,45297
7,0.4286,0.5950,0.6304,0.6122,0.5914,0.6437,0.7327,0.7781,54598
9,0.4444,0.6345,0.5993,0.6164,0.5956,0.6616,0.7333,0.7928,41870
15,0.4000,0.6380,0.6159,0.6268,0.6021,0.6860,0.7408,0.7962,41630


## 6. Baseline (ngưỡng/siêu tham số cũng chỉ chọn trên validation)
- **Lớp đa số:** luôn dự đoán lớp chiếm nhiều hơn trong fit.
- **Luật lịch sử:** dự đoán 1 nếu sản phẩm đã có purchase trong cửa sổ lịch sử (`purchases > 0`).
- **Logistic Regression (Spark MLlib):** cùng đặc trưng đã chuẩn hóa; ngưỡng xác suất chọn theo F1 validation.

In [6]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array as v2a
majority = int(round(y_fit.mean())) if y_fit.mean() != 0.5 else 0
with trace.stage("logistic_regression"):
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.0).fit(prep.transform(train.where("part = 'fit'")))
def lr_scores(df):
    pdf = lr.transform(prep.transform(df)).select("product_id", v2a("probability").getItem(1).alias("p")).orderBy("product_id").toPandas()
    return pdf["p"].to_numpy()
lr_val = lr_scores(train.where("part = 'validation'"))
lr_threshold = mc.best_threshold(y_val, lr_val, np.unique(np.quantile(lr_val, np.linspace(0, 1, 201))))
print("Logistic Regression: ngưỡng", lr_threshold, "| hệ số:", dict(zip(mc.LABEL_FEATURES, np.round(lr.coefficients.toArray(), 4))))

[20:23:09] bắt đầu: logistic_regression


26/10/08 20:23:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/10/08 20:23:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


[20:23:30] xong: logistic_regression (21097 ms)


Logistic Regression: ngưỡng 0.31986234724771323 | hệ số: {'log_views': np.float64(1.7195), 'log_carts': np.float64(0.3652), 'log_purchases': np.float64(0.6972), 'cart_rate': np.float64(-0.0297), 'purchase_rate': np.float64(-0.0184), 'log_median_price': np.float64(-0.1899), 'log_distinct_users': np.float64(-0.8801), 'recent_view_share': np.float64(0.3119)}


## 7. Đánh giá cuối trên test (một lần). Tập huấn luyện KNN cuối = fit + validation đã chuẩn hóa

In [7]:
X_model = np.vstack([X_fit, X_val]); y_model = np.concatenate([y_fit, y_val]); ids_model = np.concatenate([fit_ids, val_ids])
with trace.stage("test_predict"):
    t0 = time.perf_counter()
    test_scores = mc.knn_vote_share(X_model, y_model, X_test, best_k)
    knn_predict_ms = round((time.perf_counter() - t0) * 1000)
test_pred = (test_scores >= best_threshold).astype(int)
lr_test = lr_scores(test)
results = {
    f"KNN (K={best_k}, ngưỡng {best_threshold:.3f})": (test_pred, test_scores),
    "Baseline lớp đa số": (np.full(len(y_test), majority), np.full(len(y_test), float(majority))),
    "Baseline luật lịch sử (purchases > 0)": ((past_test > 0).astype(int), (past_test > 0).astype(float)),
    f"Logistic Regression (ngưỡng {lr_threshold:.4f})": ((lr_test >= lr_threshold).astype(int), lr_test),
}
test_table = pd.DataFrame({name: {**mc.binary_metrics(y_test, pred), "pr_auc": mc.average_precision(y_test, score)}
                           for name, (pred, score) in results.items()}).T
print(f"Test: {len(y_test)} sản phẩm, tỷ lệ dương {y_test.mean():.4f}; KNN dự đoán {knn_predict_ms} ms")
test_table[["precision", "recall", "f1", "pr_auc", "balanced_accuracy", "accuracy", "tp", "fp", "fn", "tn"]].round(4)

[20:23:33] bắt đầu: test_predict


[20:28:45] xong: test_predict (311871 ms)


Test: 64254 sản phẩm, tỷ lệ dương 0.2533; KNN dự đoán 311866 ms


,precision,recall,f1,pr_auc,balanced_accuracy,accuracy,tp,fp,fn,tn
"KNN (K=15, ngưỡng 0.400)",0.6063,0.6374,0.6214,0.6708,0.7485,0.8033,10375.0,6738.0,5903.0,41238.0
Baseline lớp đa số,0.0000,0.0000,0.0000,0.2533,0.5000,0.7467,0.0,0.0,16278.0,47976.0
Baseline luật lịch sử (purchases > 0),0.4561,0.7684,0.5725,0.4092,0.7288,0.7092,12508.0,14913.0,3770.0,33063.0
Logistic Regression (ngưỡng 0.3199),0.6061,0.6710,0.6369,0.7181,0.7615,0.8062,10923.0,7099.0,5355.0,40877.0


In [8]:
rng = np.random.default_rng(SEED)
boot = {"knn_f1": [], "knn_pr_auc": [], "rule_f1": []}
rule_pred = (past_test > 0).astype(int)
with trace.stage("bootstrap"):
    for _ in range(BOOTSTRAP):
        i = rng.integers(0, len(y_test), len(y_test))
        boot["knn_f1"].append(mc.binary_metrics(y_test[i], test_pred[i])["f1"])
        boot["knn_pr_auc"].append(mc.average_precision(y_test[i], test_scores[i]))
        boot["rule_f1"].append(mc.binary_metrics(y_test[i], rule_pred[i])["f1"])
ci = {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in boot.items()}
diff = np.array(boot["knn_f1"]) - np.array(boot["rule_f1"])
ci["knn_minus_rule_f1"] = [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))]
pd.DataFrame(ci, index=["CI 2.5%", "CI 97.5%"]).T.round(4)

[20:28:58] bắt đầu: bootstrap


[20:29:05] xong: bootstrap (7815 ms)


,CI 2.5%,CI 97.5%
knn_f1,0.6151,0.6273
knn_pr_auc,0.6633,0.6776
rule_f1,0.5669,0.5781
knn_minus_rule_f1,0.0437,0.0538


In [9]:
cm = pd.DataFrame([[int(((y_test == 0) & (test_pred == 0)).sum()), int(((y_test == 0) & (test_pred == 1)).sum())],
                   [int(((y_test == 1) & (test_pred == 0)).sum()), int(((y_test == 1) & (test_pred == 1)).sum())]],
                  index=["thực tế 0", "thực tế 1"], columns=["dự đoán 0", "dự đoán 1"])
print(f"Confusion matrix KNN trên test (K={best_k}, ngưỡng {best_threshold:.3f}):")
cm

Confusion matrix KNN trên test (K=15, ngưỡng 0.400):


,dự đoán 0,dự đoán 1
thực tế 0,41238,6738
thực tế 1,5903,10375


## 8. Lưu mô hình, metric và metadata lên HDFS; kiểm tra suy luận từ artifact (numpy, không dùng Spark)

In [10]:
from datetime import datetime, timezone
with trace.stage("write_outputs"):
    prep.write().save(mc.hdfs(f"{OUT}/preprocess_model"))                 # Spark PipelineModel (VectorAssembler + StandardScaler)
    train_pdf = pd.DataFrame({"row": np.arange(len(y_model)), "product_id": ids_model, "label": y_model})
    for j, f in enumerate(mc.LABEL_FEATURES):
        train_pdf[f"z_{f}"] = X_model[:, j]
    spark.createDataFrame(train_pdf).coalesce(1).write.mode("errorifexists").parquet(mc.hdfs(f"{OUT}/train_set"))
    spark.createDataFrame(pd.DataFrame({"product_id": test_ids, "label": y_test, "vote_share": test_scores, "prediction": test_pred})) \
        .coalesce(1).write.mode("errorifexists").parquet(mc.hdfs(f"{OUT}/test_predictions"))
model_json = {"type": "knn-classifier", "runId": RUN_ID, "labelsRunId": LABELS_RUN_ID, "k": best_k, "threshold": best_threshold,
              "features": mc.LABEL_FEATURES, "scalerMean": scaler.mean.toArray().tolist(), "scalerStd": scaler.std.toArray().tolist(),
              "distance": "squaredEuclidean (sum over features in order)", "tieBreak": "lower train row first",
              "score": "vote_share = positive neighbours / k (not a calibrated probability)",
              "trainSet": f"{OUT}/train_set", "trainRows": int(len(y_model))}
metrics_json = {"validationSweep": sweep.reset_index().to_dict(orient="records"),
                "test": {name: {k: float(v) for k, v in row.items()} for name, row in test_table.iterrows()},
                "testBootstrapCI95": ci, "classBalance": class_table.to_dict(orient="records")}
metadata_json = {"model_type": "classification", "algorithm": "k-nearest neighbours (exact, numpy)", "run_id": RUN_ID,
                 "training_time": datetime.now(timezone.utc).isoformat(), "dataset": {"tag": TAG, "labelsRunId": LABELS_RUN_ID, **labels_run["params"]},
                 "features": mc.LABEL_FEATURES, "feature_preprocessing": "log1p (Java A7) + StandardScaler fit on fit part (Spark MLlib)",
                 "label": labels_run["metrics"]["label"], "hyperparameters": {"k": best_k, "threshold": best_threshold, "maxTrain": MAX_TRAIN, "seed": SEED},
                 "metrics": {"test_f1": float(test_table.iloc[0]["f1"]), "test_pr_auc": float(test_table.iloc[0]["pr_auc"])},
                 "model_path": OUT, "status": "trained"}
LOCAL.mkdir(parents=True, exist_ok=True)
for name, value in (("model.json", model_json), ("metrics.json", metrics_json), ("metadata.json", metadata_json)):
    (LOCAL / name).write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    mc.write_hdfs_json(spark, f"{OUT}/{name}", value)
train_pdf.to_csv(LOCAL / "knn-train-set.csv", index=False)
# Suy luận từ artifact: đọc lại train_set trên HDFS, chuẩn hóa test bằng mean/std trong model.json.
saved = spark.read.parquet(mc.hdfs(f"{OUT}/train_set")).orderBy("row").toPandas()
raw_test = prep.stages[0].transform(test).select("product_id", vector_to_array("raw_features").alias("v")).orderBy("product_id").toPandas()
# Giống StandardScalerModel của Spark: (x - mean) * (1 / std).
z = (np.array(raw_test["v"].tolist()) - np.array(model_json["scalerMean"])) * (1.0 / np.array(model_json["scalerStd"]))
replay = mc.knn_vote_share(saved[[f"z_{f}" for f in mc.LABEL_FEATURES]].to_numpy(), saved["label"].to_numpy(), z, model_json["k"])
parity = float((np.abs(replay - test_scores) < 1e-12).mean())
print(f"Parity suy luận từ artifact so với đánh giá: {parity:.6f} trên {len(replay)} sản phẩm test")
assert parity == 1.0

[20:29:05] bắt đầu: write_outputs


26/10/08 20:29:13 WARN TaskSetManager: Stage 66 contains a task of very large size (5402 KiB). The maximum recommended task size is 1000 KiB.


26/10/08 20:29:18 WARN TaskSetManager: Stage 67 contains a task of very large size (1686 KiB). The maximum recommended task size is 1000 KiB.


[20:29:19] xong: write_outputs (13861 ms)


Parity suy luận từ artifact so với đánh giá: 1.000000 trên 64254 sản phẩm test


## 9. Lưu vết lần chạy

In [11]:
trace.record["environment"] = ENV
trace.record["metrics"] = {"bestK": best_k, "threshold": best_threshold, "trainRows": int(len(y_model)), "testRows": int(len(y_test)),
                           "testPositiveRate": float(y_test.mean()), "knnPredictMillis": knn_predict_ms, "artifactParity": parity,
                           "test": metrics_json["test"], "testBootstrapCI95": ci}
sweep.to_csv(LOCAL / "knn-validation-sweep.csv"); test_table.to_csv(LOCAL / "knn-test-metrics.csv"); cm.to_csv(LOCAL / "knn-confusion.csv")
record = trace.finish(LOCAL)
mc.write_hdfs_json(spark, f"{OUT}/_run.json", record)
print(json.dumps(trace.record["metrics"], indent=2, default=str))
print("Kết quả cục bộ:", LOCAL, "| HDFS:", OUT)

{
  "bestK": 15,
  "threshold": 0.4,
  "trainRows": 60876,
  "testRows": 64254,
  "testPositiveRate": 0.2533383135680269,
  "knnPredictMillis": 311866,
  "artifactParity": 1.0,
  "test": {
    "KNN (K=15, ng\u01b0\u1ee1ng 0.400)": {
      "tp": 10375.0,
      "fp": 6738.0,
      "fn": 5903.0,
      "tn": 41238.0,
      "accuracy": 0.8032651663709652,
      "precision": 0.6062642435575294,
      "recall": 0.6373633124462464,
      "f1": 0.6214249348626875,
      "balanced_accuracy": 0.7484590449174704,
      "pr_auc": 0.6707647275274352
    },
    "Baseline l\u1edbp \u0111a s\u1ed1": {
      "tp": 0.0,
      "fp": 0.0,
      "fn": 16278.0,
      "tn": 47976.0,
      "accuracy": 0.7466616864319731,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "balanced_accuracy": 0.5,
      "pr_auc": 0.2533383135680269
    },
    "Baseline lu\u1eadt l\u1ecbch s\u1eed (purchases > 0)": {
      "tp": 12508.0,
      "fp": 14913.0,
      "fn": 3770.0,
      "tn": 33063.0,
      "accura

In [12]:
spark.stop()